# Décorateurs en Python : Fonctions d’Ordre Supérieur et Plus

Dans cette section, nous allons explorer les **décorateurs**, une fonctionnalité puissante de Python basée sur les **fonctions d’ordre supérieur**. Nous aborderons leur création, leur utilisation pour modifier ou étendre le comportement des fonctions.

## Qu’est-ce qu’un Décorateur ?
Un décorateur est une fonction qui prend une autre fonction en entrée, ajoute ou modifie son comportement, et renvoie une nouvelle fonction. C’est une application directe des **fonctions d’ordre supérieur**, où les fonctions sont traitées comme des objets manipulables.

Parfois, on peut vouloir ajouter un comportement a une fonction.. sans pour autant toucher a son code.

Par exemple, on dispose d'une fonction qui trie une liste d'objet, et on veut:

- logguer l'activité
- mesurer le temps de calcul
- sécuriser la fonction
- mettre un cache afin de ne pas toujours tout recommencer
- vérifier l’accès.
- valider les arguments..
- formater le résultat...

Au lieu d'ajouter tout cela dans le code meme de notre fonction (ce qui la rendrait embrouillante)
Et bien on peut utiliser un décorateur, qui va "englober" notre fonction autours d'une autre fonction afin d'en modifier le comportement

### démonstration

Reprenons la fonction de Fibonacci que nous avons crée dans la section `Débutant` de cette formation

In [1]:
def fibonacci(n: int) -> int:
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [2]:
print(fibonacci(10)) 

55


Pour améliorer cette fonction, on pourrait vouloir mesurer son temps d'execution, enregistrer les résultats dans un cache, les afficher de maniere plus lisible, ou encore utiliser un Log.

Notre code devient alors :

In [ ]:
import time

# Un petit cache global (pas très élégant mais tu ne veux ni décorateurs ni wrappers)
_cache_fibo = {}

def fibonacci_plus(n: int) -> int:
    # Logging
    print(f"[LOG] Appel de fibonacci_plus({n})")

    # Cache
    if n in _cache_fibo:
        print("[CACHE] Valeur récupérée du cache !")
        return _cache_fibo[n]

    # Timer
    start = time.time()

    # Calcul réel
    if n <= 1:
        result = n
    else:
        a, b = 0, 1
        for _ in range(n - 1):
            a, b = b, a + b
        result = b

    end = time.time()

    # Stockage cache
    _cache_fibo[n] = result

    # Affichage "sympa"
    print("=== Résultat Fibonacci ===")
    print(f"n = {n}")
    print(f"Résultat : {result}")
    print(f"Temps d'exécution : {end - start:.6f} sec")
    print("==========================")

    return result

# Exemple
fibonacci_plus(10)
print()
fibonacci_plus(10)  # Cette fois il prendra le cache


C'est la le code que beaucoup de développeurs débutants ou intermédiaires pourraient écrire.

Parce qu'on veut bien faire (en rajoutant des détails) Notre fonction de Fibonnaci est malheureusement devenue illisible (car elle est remplie de lignes de code qui n'ont **rien a voir avec le calcul de fibonnacci!!!**)

Afin de corriger cela... Nous allons faire appel a des fonctions d'ordre supérieur

## Fonctions d’Ordre Supérieur

Une **fonction d’ordre supérieur** est une fonction qui :
- Prend une ou plusieurs fonctions comme arguments.
- Ou renvoie une fonction comme résultat.

### Exemple Simple
Illustrons ce concept avant de passer aux décorateurs.

In [ ]:
def affichage_sympa(fn, *args, **kwargs):
    """Fonction d'ordre supérieur qui affiche joliment le résultat d'une fonction."""
    result = fn(*args, **kwargs)

    print("=== Résultat ===")
    print(f"Fonction : {fn.__name__}")
    print(f"Entrées  : args={args}, kwargs={kwargs}")
    print(f"Sortie   : {result}")
    print("=================")

    return "Terminé"

In [4]:
def fibonacci(n):
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [5]:
affichage_sympa(fibonacci, 10)

=== Résultat ===
Fonction : fibonacci
Entrées  : args=(10,), kwargs={}
Sortie   : 55


'Terminé'

## Création d’un Décorateur

Un décorateur est une fonction qui enveloppe une autre fonction pour ajouter un comportement avant, après ou autour de son exécution. L'utilisation `*args` et `**kwargs` nous permet de gerer des fonctions avec des parametres

### Syntaxe
```python
def decorateur(fonction):
    def enveloppe(*args, **kwargs):
        # Avant
        resultat = fonction(*args, **kwargs)
        # Après
        return resultat
    return enveloppe

@decorateur
def ma_fonction():
    pass
```

### Exemple Simple

In [6]:
def decorateur_affichage_sympa(fn):
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)

        print("=== Résultat ===")
        print(f"Fonction : {fn.__name__}")
        print(f"Entrées  : args={args}, kwargs={kwargs}")
        print(f"Sortie   : {result}")
        print("=================")

        return "Terminé"
    return wrapper

In [7]:
@decorateur_affichage_sympa
def fibonacci(n):
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b


In [8]:
fibonacci(10)

=== Résultat ===
Fonction : fibonacci
Entrées  : args=(10,), kwargs={}
Sortie   : 55


'Terminé'

In [10]:
@decorateur_affichage_sympa
def addition(x, y):
    return x + y

In [11]:
addition(x=5, y=3)

=== Résultat ===
Fonction : addition
Entrées  : args=(), kwargs={'x': 5, 'y': 3}
Sortie   : 8


'Terminé'

## Empilement de plusieurs décorateurs
Il est bien sur possible (mais peu courant) d'ajouter autant de décorateurs qu'on le désire sur nos fonctions Python.
Voici par exemple ce que le code du début nous donne si on créer des décorateurs pour les notions de caches, de timers, etc.

In [12]:
def decorateur_cache(fn):
    cache = {}

    def wrapper(*args):
        if args in cache:
            print(f"[CACHE] {fn.__name__}{args} → récupéré du cache")
            return cache[args]

        result = fn(*args)
        cache[args] = result
        return result

    return wrapper

In [13]:
import time

def decorateur_timer(fn):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = fn(*args, **kwargs)
        end = time.time()

        print(f"[TIMER] {fn.__name__} exécutée en {end - start:.6f} sec")
        return result
    return wrapper

In [14]:
@decorateur_affichage_sympa
@decorateur_timer
@decorateur_cache
def fibonacci(n):
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [15]:
fibonacci(10)

[TIMER] wrapper exécutée en 0.000005 sec
=== Résultat ===
Fonction : wrapper
Entrées  : args=(10,), kwargs={}
Sortie   : 55


'Terminé'

In [16]:
fibonacci(10)

[CACHE] fibonacci(10,) → récupéré du cache
[TIMER] wrapper exécutée en 0.000082 sec
=== Résultat ===
Fonction : wrapper
Entrées  : args=(10,), kwargs={}
Sortie   : 55


'Terminé'

## Décorateurs avec Paramètres

Pour passer des paramètres au décorateur lui-même, ajoutez une couche de fonction supplémentaire.

### Syntaxe
```python
def decorateur_parametrable(param):
    def decorateur(fonction):
        def enveloppe(*args, **kwargs):
            # Utilise param
            return fonction(*args, **kwargs)
        return enveloppe
    return decorateur
```

### Exemple
Créons un décorateur pour indiquer si notre fonction dépasse un certain temps d'execution

In [17]:
import time

def timeout(seuil):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            start = time.time()
            result = fn(*args, **kwargs)
            end = time.time()

            duree = end - start
            if duree > seuil:
                print(f"[ALERTE] {fn.__name__} a pris {duree:.3f}s (seuil={seuil}s)")
            return result
        return wrapper
    return decorator


In [22]:
@timeout(0.001)
def fibonacci(n):
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [24]:
@decorateur_affichage_sympa
@timeout(0.001)
def addition(x, y):
    return x + y

In [25]:
addition(4, 6)

=== Résultat ===
Fonction : wrapper
Entrées  : args=(4, 6), kwargs={}
Sortie   : 10


'Terminé'

In [23]:
fibonacci(5000)

3878968454388325633701916308325905312082127714646245106160597214895550139044037097010822916462210669479293452858882973813483102008954982940361430156911478938364216563944106910214505634133706558656238254656700712525929903854933813928836378347518908762970712033337052923107693008518093849801803847813996748881765554653788291644268912980384613778969021502293082475666346224923071883324803280375039130352903304505842701147635242270210934637699104006714174883298422891491273104054328753298044273676822977244987749874555691907703880637046832794811358973739993110106219308149018570815397854379195305617510761053075688783766033667355445258844886241619210553457493675897849027988234351023599844663934853256411952221859563060475364645470760330902420806382584929156452876291575759142343809142302917491088984155209854432486594079793571316841692868039545309545388698114665082066862897420639323438488465240988742395873801976993820317174208932265468879364002630797780058759129671389634214252579116872755600360311370

## L'utilisation de `functools.wraps`

Le probleme des décorateurs, c'est qu'ils "écrasent" la fonction originale de par leur `enveloppe`, perdant ainsi les métadonnées (nom, docstring).

In [26]:
import time

def timeout(seuil):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            start = time.time()
            result = fn(*args, **kwargs)
            end = time.time()

            duree = end - start
            if duree > seuil:
                print(f"[ALERTE] {fn.__name__} a pris {duree:.3f}s (seuil={seuil}s)")
            return result
        return wrapper
    return decorator


In [27]:
@timeout(0.00005)
def fibonacci(n):
    """Calcul de la suite de Fibonacci"""
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [28]:
fibonacci.__name__

'wrapper'

In [30]:
print(fibonacci.__doc__)

None


La solution : Utiliser `wraps` de la librairie standard `functools`, qui permet de conserver les informations de la fonction englobée.

In [31]:
from functools import wraps


In [32]:
def timeout(seuil):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start = time.time()
            result = func(*args, **kwargs)
            end = time.time()

            duree = end - start
            if duree > seuil:
                print(f"[ALERTE] {func.__name__} a pris {duree:.3f}s (seuil={seuil}s)")
            return result
        return wrapper
    return decorator


In [33]:
@timeout(0.00005)
def fibonacci(n):
    """Calcul de la suite de Fibonacci"""
    if n <= 1:
        return n

    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

In [34]:
fibonacci.__name__

'fibonacci'

In [35]:
fibonacci.__doc__

'Calcul de la suite de Fibonacci'

## Utilité en Data Science et en Machine Learning

Meme si on ne les percoit pas au début, les décorateurs sont présents dans presque tous les outils que vous utiliserez en tant que Data Scientist, et nous en verrons souvent dans cette formation !

Exemples : 

- Streamlit : `@st.cache_data` et `@st.cache_resource` pour optimiser vos dashboards
- PyTorch Lightning : `@torch.no_grad()` pour désactiver les gradients
- Scikit-learn : `@property` pour créer des attributs calculés dans les transformers custom
- FastAPI : `@app.post("/predict")` pour créer des APIs
- MLflow : `@mlflow.autolog()`pour logger automatiquement les paramètres, métriques et artefacts de vos modeles

etc.


## Exercices : 

### 1. Exemple de Logging

Créez un décorateur journaliser qui, lorsqu’il est appliqué à une fonction :

1. Affiche un log avant l’exécution de la fonction, indiquant le nom de la fonction et les arguments passés.
2. Exécute la fonction.
3. Affiche un log après l’exécution de la fonction, indiquant le résultat retourné.

Ensuite, appliquez ce décorateur à une fonction multiplier(a, b) qui renvoie le produit de deux nombres entiers.

#### Correction

In [ ]:
import logging
from functools import wraps

logging.basicConfig(level=logging.INFO)


def journaliser(fonction):
    """Ajoute des logs avant et après l’exécution."""
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        logging.info(f"Appel de {fonction.__name__} avec {args}, {kwargs}")
        resultat = fonction(*args, **kwargs)
        logging.info(f"{fonction.__name__} a retourné {resultat}")
        return resultat
    return enveloppe


@journaliser
def multiplier(a: int, b: int) -> int:
    """Multiplie deux nombres."""
    return a * b


In [ ]:
print(multiplier(3, 4))

## 2. Vérification des entrées pour le calcul d'une RMSE 
Créez un décorateur `validate_numeric_inputs` qui vérifie que tous les arguments passés à une fonction `RMSE` sont des nombres (int ou float).

- Si un argument n’est pas numérique, le décorateur doit lever une TypeError avec un message clair.
- Si tous les arguments sont valides, la fonction doit s’exécuter normalement.

Appliquez ce décorateur à une fonction calculate_rmse(predictions, actuals) qui calcule le RMSE (root mean square error) entre deux listes de nombres.

Testez la fonction avec des arguments corrects et incorrects pour montrer que le décorateur fonctionne correctement.

#### Correction

In [ ]:
def validate_numeric_inputs(func):
    """Vérifie que tous les arguments sont numériques"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if not isinstance(arg, (int, float)):
                raise TypeError(f"Tous les arguments doivent être numériques. Reçu: {type(arg)}")
        for value in kwargs.values():
            if not isinstance(value, (int, float)):
                raise TypeError(f"Tous les arguments doivent être numériques. Reçu: {type(value)}")
        return func(*args, **kwargs)
    return wrapper

In [ ]:
@validate_numeric_inputs
def calculate_rmse(predictions, actuals):
    """Calcule le RMSE entre prédictions et valeurs réelles"""
    n = len(predictions)
    squared_errors = [(p - a) ** 2 for p, a in zip(predictions, actuals)]
    return (sum(squared_errors) / n) ** 0.5


try:
    rmse = calculate_rmse([1, 2, 3], [1.1, 2.2, 2.9])
    print(f"RMSE: {rmse:.4f}")
    # Ceci va lever une erreur
    rmse = calculate_rmse("invalid", [1, 2, 3])
except TypeError as e:
    print(f"Erreur attrapée: {e}")

## Exercice 3

Créer une fonction qui fournie un nombre aléatoire entre 0 et 10
ainsi qu'un décorateur `retry` qui permet de réessayer l'operation un certain nombre de fois, si la valeur obtenue est en dessous d'un certain seuil.

#### Correction

In [ ]:
import random
from functools import wraps

def retry(max_tries: int, seuil: float):
    """Décorateur qui réessaie la fonction tant que la valeur est < seuil."""
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            attempt = 1
            result = fn(*args, **kwargs)

            while result < seuil and attempt < max_tries:
                print(f"[Retry] Essai {attempt}/{max_tries} : valeur {result} < seuil {seuil}")
                attempt += 1
                result = fn(*args, **kwargs)

            print(f"[Final] Valeur obtenue : {result}")
            return result

        return wrapper
    return decorator


@retry(max_tries=5, seuil=7)
def nombre_aleatoire():
    return random.randint(0, 10)


# Exemple d'utilisation
nombre_aleatoire()

## Conclusion

Cette section vous a permis de maîtriser :
- Les **fonctions d’ordre supérieur** comme base des décorateurs.
- La **création et utilisation de décorateurs** pour ajouter des comportements (logs, validation, temps)

Les Cas d'usage courrant : 


1. Performance monitoring (timeit)
2. Logging et debugging
3. Validation des données
4. Caching et optimisation
5. Retry logic et gestion d'erreurs
6. Design patterns (singleton, factory)
7. Composition de décorateurs
8. Intégration avec frameworks DS/ML
9. Tracking d'expériences
10. Gestion des ressources (GPU/CPU)


Les décorateurs sont un outil élégant pour étendre vos fonctions sans les modifier directement. Expérimentez avec ces exemples pour créer vos propres décorateurs adaptés à vos besoins !